# Linear Layer

全连接层的核心计算：

$$y = xW^T + b$$

- $x$ — 输入，shape `(B, in_features)`
- $W$ — 权重矩阵，shape `(out_features, in_features)`
- $b$ — 偏置，shape `(out_features,)`
- $y$ — 输出，shape `(B, out_features)`

## 权重初始化

权重用 **Kaiming（He）初始化**的简化版：

$$W \sim \mathcal{N}\left(0,\ \frac{1}{\sqrt{\text{in\_features}}}\right)$$

目的是让每一层输出的方差保持稳定，不随网络加深而爆炸或消失。

偏置初始化为 0，这是标准做法——权重负责打破对称性，偏置不需要随机。

In [ ]:
import torch
import math

In [ ]:
class SimpleLinear:
    def __init__(self, in_features: int, out_features: int):
        self.weight = torch.randn(out_features, in_features) * (1 / math.sqrt(in_features))  # (out, in)
        self.weight.requires_grad_(True)   # 标记为可训练参数
        self.bias = torch.zeros(out_features, requires_grad=True)  # 偏置初始化为 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight.T + self.bias  # (B, in) @ (in, out) + (out,) -> (B, out)

## 验证

In [ ]:
layer = SimpleLinear(8, 4)
print("W shape:", layer.weight.shape)   # (4, 8)
print("b shape:", layer.bias.shape)     # (4,)

x = torch.randn(2, 8)                  # batch=2, in_features=8
print("Output shape:", layer.forward(x).shape)  # (2, 4)

## 为什么 W 的 shape 是 (out, in) 而不是 (in, out)？

`nn.Linear` 存的是 `(out_features, in_features)`，forward 时转置再乘：

```
x @ W.T
(B, in) @ (in, out) -> (B, out)
```

这样存储的好处是：每一行是一个输出神经元的权重向量，在做梯度更新和参数访问时内存更连续。